In [1]:
import torch
import torch.nn as nn


# fix seed so that random initialization always performs the same
torch.manual_seed(13)


# create the model N as described in the question
N = nn.Sequential(nn.Linear(10, 10, bias=False),
                  nn.ReLU(),
                  nn.Linear(10, 10, bias=False),
                  nn.ReLU(),
                  nn.Linear(10, 3, bias=False))

# random input
x = torch.rand((1,10)) # the first dimension is the batch size; the following dimensions the actual dimension of the data
x.requires_grad_() # this is required so we can compute the gradient w.r.t x

t = 1 # target class

epsReal = 1.15 #depending on your data this might be large or small
eps = epsReal - 1e-7 # small constant to offset floating-point erros

# The network N classfies x as belonging to class 2
original_class = N(x).argmax(dim=1).item()  # TO LEARN: make sure you understand this expression
print("Original Class: ", original_class)
assert(original_class == 2)

# compute gradient
# note that CrossEntropyLoss() combines the cross-entropy loss and an implicit softmax function
L = nn.CrossEntropyLoss()
loss = L(N(x), torch.tensor([t], dtype=torch.long)) # TO LEARN: make sure you understand this line
loss.backward()

# Take the gradient with respect to x... clip according to our budget... then add to input (mult by eps and sign)
x_new = x - (eps * torch.sign(x.grad))
x_new = torch.clamp(x_new, x_new - eps, x_new + eps)
new_class = N(x_new).argmax(dim=1).item()  # TO LEARN: make sure you understand this expression
print("New Class: ", new_class)

print("Norm Diff: ", torch.linalg.vector_norm(x, ord = 2, dim = 1, keepdim = True) - torch.linalg.vector_norm(x_new, ord = 2, dim = 1, keepdim = True))


Original Class:  2
New Class:  1
Norm Diff:  tensor([[-2.0067]], grad_fn=<SubBackward0>)


In [2]:
torch.linalg.vector_norm(x)

tensor(2.0556, grad_fn=<LinalgVectorNormBackward0>)

In [3]:
torch.linalg.vector_norm(x_new)

tensor(4.0623, grad_fn=<LinalgVectorNormBackward0>)